#Group Diversity Analysis

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/Datathon_Group_Diversity/

In [ ]:
file_path = "/content/drive/MyDrive/Datathon_Group_Diversity/Manuscript_Teams.csv"
df_raw = pd.read_csv(file_path, encoding='utf-8')

In [ ]:
df_raw.head()

In [ ]:
import numpy as np

initial_count = df_raw.shape[0]
print(f"Initial row count: {initial_count}")

invalid_values = {
    '-', '–', '-', '', 'nan', 'none', 'na', 'n/a'
}

mask = ~df_raw['Unnamed: 2'].astype(str).str.strip().str.lower().isin(invalid_values)

df_filtered = df_raw[mask].copy()

final_count = df_filtered.shape[0]
print(f"Row count after dropping empty 'Unnamed: 2' rows: {final_count}")
print(f"Number of rows dropped: {initial_count - final_count}")

In [ ]:
collab_df = df_raw.copy()

collab_df['Unnamed: 2'] = (
    collab_df['Unnamed: 2']
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_team_ids = {
    '-', '–', '-', '', 'nan', 'none', 'na', 'n/a'
}

collab_df = collab_df[
    ~collab_df['Unnamed: 2'].isin(invalid_team_ids)
].copy()

collab_df['Unnamed: 2'] = collab_df['Unnamed: 2'].str.upper()

In [ ]:
collab_df.head()

In [ ]:
import pandas as pd
import numpy as np
import re

collab_df = df_raw.copy()

needed_cols = [
    'Unnamed: 2',
    'Email Address',
    'Current Institution',
    'Year in Program',
    'Degree Program(s)'
]

collab_df = collab_df[needed_cols].copy()
collab_df.columns = [
    'team_id_raw',
    'participant_id_raw',
    'institution_raw',
    'year_raw',
    'degree_raw'
]

collab_df['participant_id'] = (
    collab_df['participant_id_raw']
    .astype(str)
    .str.strip()
    .str.lower()
)

collab_df['team_id'] = (
    collab_df['team_id_raw']
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_team_ids = {
    '-', '–', '—', '', 'nan', 'none', 'na', 'n/a'
}

collab_df = collab_df[
    ~collab_df['team_id'].isin(invalid_team_ids)
].copy()

collab_df['team_id'] = collab_df['team_id'].str.upper()

def normalize_training_level(val):
    if pd.isna(val):
        return np.nan

    txt = str(val).strip().lower()

    if txt in ['6+', '6 +', '6plus', '6 plus']:
        return '6+'

    try:
        num = int(float(txt))
        if num >= 6:
            return '6+'
        if 1 <= num <= 5:
            return str(num)
    except ValueError:
        pass

    return np.nan

collab_df['training_level'] = collab_df['year_raw'].apply(normalize_training_level)

def normalize_degree(cell):
    if pd.isna(cell):
        return np.nan

    txt = str(cell).lower()
    txt = re.sub(r'[^\w\s/]', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt)

    has_md = bool(re.search(r'\b(md|mbbs|mbchb|m\.b\.ch\.b)\b', txt))
    has_do = bool(re.search(r'\bdo\b', txt))
    has_dds = bool(re.search(r'\b(dds|dmd|bds)\b', txt))
    has_phd = bool(re.search(r'\bphd\b', txt))
    has_masters = bool(re.search(r'\b(ms|msc|mph|mba|mhs|master)\b', txt))
    has_bachelors = bool(re.search(r'\b(bs|ba|bsc|bachelor|undergraduate)\b', txt))

    if has_md and (has_phd or has_masters):
        return "Dual MD"
    if has_md:
        return "MD"
    if has_do:
        return "DO"
    if has_dds:
        return "DDS"
    if has_phd:
        return "PhD"
    if has_masters:
        return "Masters"
    if has_bachelors:
        return "Bachelors"

    return np.nan

collab_df['degree_category'] = collab_df['degree_raw'].apply(normalize_degree)

inst_clean = (
    collab_df['institution_raw']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

INSTITUTION_MAP = {
    r'\butsw\b|ut southwestern|university of texas southwestern':
        'University of Texas Southwestern',
    r'icahn.*mount sinai|mount sinai|mssm':
        'Icahn School of Medicine at Mount Sinai',
    r'warren alpert|brown university':
        'Brown University',
    r'ucla.*geffen|ucla school of medicine|^ucla$':
        'UCLA David Geffen School of Medicine',
    r'university of miami|miller school':
        'University of Miami Miller School of Medicine',
    r'stanford.*medicine|^stanford$':
        'Stanford School of Medicine',
    r'yale.*medicine':
        'Yale School of Medicine',
    r'duke.*medicine|^duke university$':
        'Duke University School of Medicine',
    r'emory.*(school|university)':
        'Emory University School of Medicine'
}

def canonicalize_institution(inst):
    if inst in ['nan', 'none', 'graduated', 'graduate', '']:
        return np.nan

    for pattern, canonical in INSTITUTION_MAP.items():
        if re.search(pattern, inst):
            return canonical

    return inst.title()

collab_df['institution'] = inst_clean.apply(canonicalize_institution)

team_tbl = collab_df.groupby('team_id').agg(
    n_participants=('participant_id', 'nunique'),
    institutions=('institution', lambda s: sorted(s.dropna().unique())),
    training_levels=('training_level', lambda s: sorted(s.dropna().unique())),
    degree_types=('degree_category', lambda s: sorted(s.dropna().unique()))
).reset_index()

team_tbl['n_institutions'] = team_tbl['institutions'].apply(len)
team_tbl['n_training_levels'] = team_tbl['training_levels'].apply(len)
team_tbl['n_degree_types'] = team_tbl['degree_types'].apply(len)

print("Team size distribution:")
print(
    collab_df
    .groupby('team_id')['participant_id']
    .nunique()
    .value_counts()
    .sort_index()
)

print("\nHead of team table:")
print(team_tbl.head())

audit_institutions = (
    collab_df[['institution_raw', 'institution']]
    .drop_duplicates()
    .sort_values('institution')
)

audit_degrees = (
    collab_df[['degree_raw', 'degree_category']]
    .drop_duplicates()
    .sort_values('degree_category')
)

print(team_tbl.head())
print(team_tbl[['n_institutions', 'n_training_levels', 'n_degree_types']].describe())

In [ ]:
team_tbl.head(150)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

n_teams = team_tbl.shape[0]

pct_multi_inst = (
    (team_tbl['n_institutions'] >= 2).mean() * 100
)

mean_inst = team_tbl['n_institutions'].mean()
median_inst = team_tbl['n_institutions'].median()

inst_dist = (
    team_tbl['n_institutions']
    .value_counts()
    .sort_index()
)

print(f"% teams with ≥2 institutions: {pct_multi_inst:.1f}%")
print(f"Mean institutions per team: {mean_inst:.2f}")
print(f"Median institutions per team: {median_inst:.1f}")
print(inst_dist)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(
    x='n_institutions',
    data=team_tbl,
    color='steelblue'
)
plt.xlabel('Number of Institutions per Team')
plt.ylabel('Number of Teams')
plt.title('Distribution of Cross-Institution Collaboration')
plt.tight_layout()
plt.show()

In [ ]:
pct_multi_degree = (
    (team_tbl['n_degree_types'] >= 2).mean() * 100
)

print(f"% teams with ≥2 degree types: {pct_multi_degree:.1f}%")

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(
    x='n_degree_types',
    data=team_tbl,
    color='darkorange'
)
plt.xlabel('Number of Degree Types per Team')
plt.ylabel('Number of Teams')
plt.title('Cross-Degree Collaboration')
plt.tight_layout()
plt.show()

In [ ]:
joint_matrix = (
    team_tbl
    .groupby(['n_institutions', 'n_training_levels'])
    .size()
    .unstack(fill_value=0)
)

joint_matrix

In [ ]:
plt.figure(figsize=(7,5))
sns.heatmap(
    joint_matrix,
    annot=True,
    fmt='d',
    cmap='Blues'
)
plt.xlabel('Number of Training Levels per Team')
plt.ylabel('Number of Institutions per Team')
plt.title('Joint Institutional and Training-Level Diversity')
plt.tight_layout()
plt.show()

In [ ]:
def shannon_entropy(values):
    vals = [v for v in values if pd.notna(v)]
    if len(vals) == 0:
        return np.nan
    vc = pd.Series(vals).value_counts()
    p = vc / vc.sum()
    return float(-(p * np.log(p)).sum())

def simpson_index(values):
    vals = [v for v in values if pd.notna(v)]
    if len(vals) == 0:
        return np.nan
    vc = pd.Series(vals).value_counts()
    p = vc / vc.sum()
    return float(1 - (p * p).sum())


team_entropy_inst = (
    collab_df
    .groupby('team_id')['institution']
    .apply(lambda s: shannon_entropy(s.dropna().tolist()))
)

team_entropy_train = (
    collab_df
    .groupby('team_id')['training_level']
    .apply(lambda s: shannon_entropy(s.dropna().tolist()))
)

team_tbl = team_tbl.merge(
    team_entropy_inst.rename('entropy_institution'),
    on='team_id'
)

team_tbl = team_tbl.merge(
    team_entropy_train.rename('entropy_training_level'),
    on='team_id'
)

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(
    team_tbl['entropy_institution'],
    bins=20,
    kde=True,
    color='purple'
)
plt.xlabel('Shannon Entropy (Institutions)')
plt.title('Institutional Diversity Across Teams')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(
    team_tbl['entropy_training_level'],
    bins=20,
    kde=True,
    color='green'
)
plt.xlabel('Shannon Entropy (Training Level)')
plt.title('Training-Level Diversity Across Teams')
plt.tight_layout()
plt.show()

In [ ]:
team_simpson_inst = (
    collab_df
    .groupby('team_id')['institution']
    .apply(lambda s: simpson_index(s.dropna().tolist()))
)

team_simpson_train = (
    collab_df
    .groupby('team_id')['training_level']
    .apply(lambda s: simpson_index(s.dropna().tolist()))
)

team_tbl = team_tbl.merge(
    team_simpson_inst.rename('simpson_institution'),
    on='team_id'
)

team_tbl = team_tbl.merge(
    team_simpson_train.rename('simpson_training_level'),
    on='team_id'
)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(
    team_tbl['entropy_institution'],
    team_tbl['simpson_institution'],
    alpha=0.7
)
plt.xlabel('Shannon Entropy (Institution)')
plt.ylabel('Simpson Index (Institution)')
plt.title('Consistency of Diversity Metrics')
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx


B = nx.Graph()

for _, row in collab_df.iterrows():
    p = f"participant_{row['participant_id']}"
    i = f"institution_{row['institution']}"

    if pd.notna(row['institution']):
        B.add_node(p, bipartite='participant')
        B.add_node(i, bipartite='institution')
        B.add_edge(p, i)

In [ ]:
inst_nodes = [
    n for n, d in B.nodes(data=True)
    if d.get('bipartite') == 'institution'
]

G_inst = nx.bipartite.weighted_projected_graph(
    B, inst_nodes
)

density = nx.density(G_inst)
betweenness = nx.betweenness_centrality(G_inst, weight='weight')

print(f"Institution network density: {density:.3f}")

In [ ]:
plt.figure(figsize=(8,8))
pos = nx.spring_layout(G_inst, seed=42, k=0.5)

edge_weights = [G_inst[u][v]['weight'] for u, v in G_inst.edges()]

nx.draw_networkx_nodes(
    G_inst, pos,
    node_size=600,
    node_color='lightblue'
)
nx.draw_networkx_edges(
    G_inst, pos,
    width=[w * 0.5 for w in edge_weights],
    alpha=0.7
)
nx.draw_networkx_labels(
    G_inst, pos,
    {n: n.replace('institution_', '') for n in G_inst.nodes()},
    font_size=8
)

plt.title('Institution-Level Collaboration Network')
plt.axis('off')
plt.tight_layout()
plt.show()

#Thematic Analysis

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Datathon_Group_Diversity/Thematic_Data_2.csv"
df = pd.read_csv(file_path, encoding='utf-8')

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

tqdm.pandas()
print(df.head())
print(df.columns.tolist())
print(df.shape)

review_df = df.copy()

In [ ]:
cols_to_show = ['Free response','Kaden Codes','Kaden Thematic','Sam Codes','Sam Thematic','Mitali Codes','Mitali Thematic']
review_data = review_df.iloc[2:].copy().reset_index(drop=True)
review_data['source_row'] = review_data.index + 2
print(review_data[cols_to_show].head(12))
print(review_data[cols_to_show].tail(8))

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

tqdm.pandas()
working_df = review_df.iloc[2:].copy().reset_index(drop=True)
working_df['free_response_clean'] = working_df['Free response'].fillna('').astype(str).str.replace('_', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()
working_df['free_response_norm'] = working_df['free_response_clean'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
working_df['word_count'] = working_df['free_response_clean'].str.split().str.len()
working_df['is_blank'] = working_df['free_response_clean'].eq('') | working_df['free_response_clean'].str.lower().isin(['nan','none'])
working_df['is_duplicate'] = working_df['free_response_norm'].duplicated(keep='first')
working_df['is_nonsense'] = working_df['word_count'].fillna(0).lt(3)
usable_review_df = working_df[(~working_df['is_blank']) & (~working_df['is_duplicate']) & (~working_df['is_nonsense'])].copy().reset_index(drop=True)
usable_review_df['response_id'] = usable_review_df.index + 1
print(working_df[['free_response_clean','is_blank','is_duplicate','is_nonsense']].head(12))
print(working_df[['is_blank','is_duplicate','is_nonsense']].sum())
print(len(usable_review_df))
print(usable_review_df[['response_id','free_response_clean','Kaden Codes','Kaden Thematic','Sam Codes','Sam Thematic']].head(10))

In [ ]:
for col_val in ['Kaden Codes','Kaden Thematic','Sam Codes','Sam Thematic']:
    print(col_val)
    print(usable_review_df[col_val].dropna().astype(str).str.strip().value_counts().head(20))
    print('---')

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

def split_codes(cell_val):
    if pd.isna(cell_val):
        return []
    text_val = str(cell_val).strip()
    if text_val.lower() in ['none', 'nan', '']:
        return []
    text_val = text_val.replace('‘', '').replace('’', '').replace("'", '').replace('“', '').replace('”', '')
    text_val = text_val.replace(';', ',')
    parts = [re.sub(r'\s+', ' ', part.strip().lower()) for part in text_val.split(',')]
    parts = [part for part in parts if part and part not in ['none', 'nan']]
    fixes = {'cross-time zone coordination challeneges':'cross-time zone coordination challenges'}
    return [fixes.get(part, part) for part in parts]

if 'usable_review_df' not in globals():
    working_df = review_df.iloc[2:].copy().reset_index(drop=True)
    working_df['free_response_clean'] = working_df['Free response'].fillna('').astype(str).str.replace('_', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()
    working_df['free_response_norm'] = working_df['free_response_clean'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
    working_df['word_count'] = working_df['free_response_clean'].str.split().str.len()
    working_df['is_blank'] = working_df['free_response_clean'].eq('') | working_df['free_response_clean'].str.lower().isin(['nan','none'])
    working_df['is_duplicate'] = working_df['free_response_norm'].duplicated(keep='first')
    working_df['is_nonsense'] = working_df['word_count'].fillna(0).lt(3)
    usable_review_df = working_df[(~working_df['is_blank']) & (~working_df['is_duplicate']) & (~working_df['is_nonsense'])].copy().reset_index(drop=True)
    usable_review_df['response_id'] = usable_review_df.index + 1

usable_review_df['kaden_codes_list'] = usable_review_df['Kaden Codes'].apply(split_codes)
usable_review_df['sam_codes_list'] = usable_review_df['Sam Codes'].apply(split_codes)
usable_review_df['combined_codes_list'] = usable_review_df.apply(lambda row: sorted(set(row['kaden_codes_list'] + row['sam_codes_list'])), axis=1)

print('usable responses')
print(len(usable_review_df))
print('kaden coded rows')
print((usable_review_df['kaden_codes_list'].str.len() > 0).sum())
print('sam coded rows')
print((usable_review_df['sam_codes_list'].str.len() > 0).sum())
print(usable_review_df[['response_id','kaden_codes_list','sam_codes_list']].head(5))

In [ ]:
from collections import Counter
import pandas as pd

kaden_counter = Counter([code for codes in usable_review_df['kaden_codes_list'] for code in codes])
sam_counter = Counter([code for codes in usable_review_df['sam_codes_list'] for code in codes])
combined_counter = Counter([code for codes in usable_review_df['combined_codes_list'] for code in codes])

kaden_code_counts = pd.DataFrame(kaden_counter.items(), columns=['code','n']).sort_values(['n','code'], ascending=[False, True]).reset_index(drop=True)
sam_code_counts = pd.DataFrame(sam_counter.items(), columns=['code','n']).sort_values(['n','code'], ascending=[False, True]).reset_index(drop=True)
combined_code_counts = pd.DataFrame(combined_counter.items(), columns=['code','n']).sort_values(['n','code'], ascending=[False, True]).reset_index(drop=True)

print(kaden_code_counts.head(40))
print(sam_code_counts.head(60))
print(combined_code_counts.head(80))

In [ ]:
def sentiment_tokens(cell_val):
    if pd.isna(cell_val):
        return []
    text_val = str(cell_val).strip().lower()
    if text_val in ['', 'none', 'nan']:
        return []
    return sorted(set([part.strip() for part in text_val.split(',') if part.strip()]))

usable_review_df['kaden_sentiment_tokens'] = usable_review_df['Kaden Thematic'].apply(sentiment_tokens)
usable_review_df['sam_sentiment_tokens'] = usable_review_df['Sam Thematic'].apply(sentiment_tokens)
usable_review_df['all_sentiment_tokens'] = usable_review_df.apply(lambda row: sorted(set(row['kaden_sentiment_tokens'] + row['sam_sentiment_tokens'])), axis=1)

def combined_sentiment(row):
    kaden_vals = set(row['kaden_sentiment_tokens'])
    sam_vals = set(row['sam_sentiment_tokens'])
    all_vals = set(row['all_sentiment_tokens'])
    if all_vals == {'positive'}:
        return 'strongly positive'
    if 'positive' in all_vals and 'neutral' in all_vals:
        return 'moderately positive'
    if all_vals == {'neutral'}:
        return 'neutral/mixed'
    return 'uncoded'

usable_review_df['combined_sentiment'] = usable_review_df.apply(combined_sentiment, axis=1)
print(usable_review_df[['response_id','Kaden Thematic','Sam Thematic','combined_sentiment']])
print(usable_review_df['combined_sentiment'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import re
from itertools import combinations

n_usable = len(usable_review_df)

def code_to_themes(code_val):
    code_text = str(code_val).lower().strip()
    themes = set()
    if any(term in code_text for term in ['teamwork','collaborat','interdisciplinary','diverse skill','diverse strength','diverse experiences','multiple perspectives','peer learning','communication within team','strong team dynamic','cohesive','coordination across roles','playing to strengths','leveraging strengths','friends working together','easy to schedule team meetings','effective coordination']):
        themes.add('Teamwork and interdisciplinarity')
    if any(term in code_text for term in ['skill','learning','knowledge','ai exposure','ml','machine learning','technical learning','educational','beginner growth','confidence','tools','exploration of ai','hands on','new domain','learn from']):
        themes.add('Learning and skill building')
    if any(term in code_text for term in ['real world','real-world','patient','clinical','problem solving','prototype','meaningful project','meaningful application','meaningful work','feasible solution','solution','practical application','clear problem','advanced analytics','creative challenge','engaging problem','focused problem','impact']):
        themes.add('Real-world problem solving and impact')
    if any(term in code_text for term in ['resource','support','mentor','feedback','examples','dataset','mimic','leadership check','guidance','accessible support','clinical mentors','explanation of datasets']):
        themes.add('Program resources and guidance')
    if any(term in code_text for term in ['communication from organizers','improved communication','deadline','deliverable','submission','onboarding','more time','timeline','expectation','format','writing burden','cumbersome','instructions','clear guidance','confusion']):
        themes.add('Organizational clarity and logistics')
    if any(term in code_text for term in ['enjoy','fun','positive experience','content with experience','networking','relationships','community','career inspiration','productive experience','appreciated short term','excitement']):
        themes.add('Community, enjoyment, and motivation')
    if any(term in code_text for term in ['data limitations','problem framing','aligning','time zone','timezone','cross-time','small team','data-heavy','scope','scoping']):
        themes.add('Project execution constraints')
    return sorted(themes)

usable_review_df['combined_themes'] = usable_review_df['combined_codes_list'].apply(lambda codes: sorted(set(theme for code in codes for theme in code_to_themes(code))))

theme_order = [
    'Teamwork and interdisciplinarity',
    'Learning and skill building',
    'Real-world problem solving and impact',
    'Program resources and guidance',
    'Organizational clarity and logistics',
    'Community, enjoyment, and motivation',
    'Project execution constraints'
]

theme_rows = []
for theme_name in theme_order:
    count_val = usable_review_df['combined_themes'].apply(lambda vals: theme_name in vals).sum()
    theme_rows.append({'theme': theme_name, 'responses': int(count_val), 'percent': round(100 * count_val / n_usable, 1)})
updated_theme_summary = pd.DataFrame(theme_rows).sort_values(['responses','theme'], ascending=[False, True]).reset_index(drop=True)

sentiment_order = ['strongly positive','moderately positive','neutral/mixed']
updated_sentiment_summary = usable_review_df['combined_sentiment'].value_counts().reindex(sentiment_order, fill_value=0).reset_index()
updated_sentiment_summary.columns = ['sentiment','responses']
updated_sentiment_summary['percent'] = (100 * updated_sentiment_summary['responses'] / n_usable).round(1)

cooc_matrix_updated = pd.DataFrame(0, index=theme_order, columns=theme_order)
for theme_vals in usable_review_df['combined_themes']:
    unique_vals = sorted(set(theme_vals))
    for theme_a in unique_vals:
        for theme_b in unique_vals:
            if theme_a != theme_b:
                cooc_matrix_updated.loc[theme_a, theme_b] = cooc_matrix_updated.loc[theme_a, theme_b] + 1

sent_by_theme = usable_review_df.explode('combined_themes').groupby(['combined_themes','combined_sentiment']).size().reset_index(name='count')
full_idx = pd.MultiIndex.from_product([theme_order, sentiment_order], names=['theme','sentiment'])
sent_by_theme = sent_by_theme.set_index(['combined_themes','combined_sentiment']).reindex(full_idx, fill_value=0).reset_index()
sent_by_theme.columns = ['theme','sentiment','count']

print(updated_theme_summary)
print(updated_sentiment_summary)
print(cooc_matrix_updated)
print(sent_by_theme)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

plot_theme_summary = updated_theme_summary.copy()
plot_theme_summary['label_text'] = plot_theme_summary['responses'].astype(str) + ' of ' + str(n_usable) + ' (' + plot_theme_summary['percent'].astype(str) + '%)'

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_theme_summary, x='responses', y='theme', color='#4C78A8')
for idx_val, row_val in plot_theme_summary.reset_index(drop=True).iterrows():
    plt.text(row_val['responses'] + 0.15, idx_val, row_val['label_text'], va='center', fontsize=11)
plt.xlabel('Number of responses')
plt.ylabel('Theme')
plt.title('Theme prevalence in participant reflections')
plt.xlim(0, max(plot_theme_summary['responses']) + 4)
plt.tight_layout()
plt.savefig('updated_figure1_theme_prevalence.png', bbox_inches='tight')
plt.show()

heat_plot = cooc_matrix_updated.loc[theme_order, theme_order]
plt.figure(figsize=(11.5, 9))
sns.heatmap(heat_plot, annot=True, fmt='d', cmap='Blues', linewidths=0.5, linecolor='white', cbar_kws={'label':'Co-occurring responses'})
plt.title('Co-occurrence of major themes across responses')
plt.xlabel('Theme')
plt.ylabel('Theme')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('updated_figure2_theme_cooccurrence_heatmap.png', bbox_inches='tight')
plt.show()

sentiment_pivot = sent_by_theme.pivot(index='theme', columns='sentiment', values='count').loc[theme_order]
sentiment_pivot = sentiment_pivot[['strongly positive','moderately positive','neutral/mixed']]
ax_obj = sentiment_pivot.plot(kind='barh', stacked=True, figsize=(12, 7.5), color=['#2E8B57', '#E3A008', '#C44E52'], edgecolor='none')
ax_obj.set_xlabel('Number of theme-coded responses')
ax_obj.set_ylabel('Theme')
ax_obj.set_title('Sentiment distribution within each theme')
ax_obj.legend(title='Sentiment', loc='lower right', frameon=True)
for container_val in ax_obj.containers:
    labels_val = [str(int(val)) if val > 0 else '' for val in container_val.datavalues]
    ax_obj.bar_label(container_val, labels=labels_val, label_type='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('updated_figure3_sentiment_by_theme.png', bbox_inches='tight')
plt.show()

print('updated_figure1_theme_prevalence.png')
print('updated_figure2_theme_cooccurrence_heatmap.png')
print('updated_figure3_sentiment_by_theme.png')

##Optimize categories

In [ ]:
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import re

all_code_rows = []
for row_val in usable_review_df.itertuples(index=False):
    for code_val in row_val.kaden_codes_list:
        all_code_rows.append({'response_id': row_val.response_id, 'reviewer': 'Kaden', 'code': code_val})
    for code_val in row_val.sam_codes_list:
        all_code_rows.append({'response_id': row_val.response_id, 'reviewer': 'Sam', 'code': code_val})

code_long_df = pd.DataFrame(all_code_rows)
code_long_df['code_clean'] = code_long_df['code'].str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)
code_freq_df = code_long_df.groupby('code_clean').agg(
    frequency=('response_id', 'count'),
    responses=('response_id', lambda vals: len(set(vals))),
    reviewers=('reviewer', lambda vals: ', '.join(sorted(set(vals))))
).reset_index().sort_values(['responses','frequency','code_clean'], ascending=[False, False, True]).reset_index(drop=True)

print(code_freq_df.head(30))
print('unique codes')
print(len(code_freq_df))
print('total code assignments')
print(len(code_long_df))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from scipy import sparse
from tqdm import tqdm

unique_codes_df = code_freq_df.copy().reset_index(drop=True)
unique_codes_df['code_id'] = unique_codes_df.index
code_texts = unique_codes_df['code_clean'].tolist()

text_vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=1, stop_words='english')
text_X = text_vectorizer.fit_transform(code_texts)

resp_code = code_long_df[['response_id','code_clean']].drop_duplicates().copy()
response_ids = sorted(resp_code['response_id'].unique())
resp_index = {resp: idx for idx, resp in enumerate(response_ids)}
code_index = {code: idx for idx, code in enumerate(code_texts)}
rows = resp_code['code_clean'].map(code_index).to_numpy()
cols = resp_code['response_id'].map(resp_index).to_numpy()
data_vals = np.ones(len(resp_code))
cooc_X = sparse.csr_matrix((data_vals, (rows, cols)), shape=(len(code_texts), len(response_ids)))

combined_X = sparse.hstack([normalize(text_X) * 0.70, normalize(cooc_X) * 0.30]).tocsr()
svd_components = min(25, combined_X.shape[0] - 1, combined_X.shape[1] - 1)
svd_model = TruncatedSVD(n_components=svd_components, random_state=42)
combined_reduced = svd_model.fit_transform(combined_X)
combined_reduced = normalize(combined_reduced)

print('unique codes')
print(len(unique_codes_df))
print('text features')
print(text_X.shape)
print('response co-occurrence features')
print(cooc_X.shape)
print('reduced matrix')
print(combined_reduced.shape)
print('variance retained')
print(round(float(svd_model.explained_variance_ratio_.sum()), 3))

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import pandas as pd
import numpy as np
from tqdm import tqdm

cluster_X = np.asarray(combined_reduced)
k_values = list(range(3, 13))
cluster_results = []
labels_store = {}

for k_val in tqdm(k_values):
    methods = {
        'kmeans': KMeans(n_clusters=k_val, n_init=50, random_state=42).fit_predict(cluster_X),
        'agg_ward': AgglomerativeClustering(n_clusters=k_val, linkage='ward').fit_predict(cluster_X),
        'agg_average': AgglomerativeClustering(n_clusters=k_val, linkage='average', metric='cosine').fit_predict(cluster_X),
        'spectral': SpectralClustering(n_clusters=k_val, affinity='nearest_neighbors', n_neighbors=10, random_state=42).fit_predict(cluster_X),
        'gmm': GaussianMixture(n_components=k_val, covariance_type='full', random_state=42).fit_predict(cluster_X)
    }
    for method_name, labels_val in methods.items():
        key_val = method_name + '_' + str(k_val)
        labels_store[key_val] = labels_val
        sil_val = silhouette_score(cluster_X, labels_val, metric='cosine')
        ch_val = calinski_harabasz_score(cluster_X, labels_val)
        db_val = davies_bouldin_score(cluster_X, labels_val)
        cluster_results.append({'method': method_name, 'k': k_val, 'silhouette_cosine': sil_val, 'calinski_harabasz': ch_val, 'davies_bouldin': db_val})

cluster_eval_df = pd.DataFrame(cluster_results)
consensus_by_k = cluster_eval_df.groupby('k').agg(
    mean_silhouette=('silhouette_cosine','mean'),
    median_silhouette=('silhouette_cosine','median'),
    mean_calinski=('calinski_harabasz','mean'),
    mean_davies=('davies_bouldin','mean')
).reset_index()
print(cluster_eval_df.sort_values('silhouette_cosine', ascending=False).head(15))
print(consensus_by_k)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering
from collections import defaultdict

candidate_ks = [6, 7, 8, 10, 12]
cluster_summary_rows = []
cluster_members = {}
for k_val in candidate_ks:
    labels_val = KMeans(n_clusters=k_val, n_init=100, random_state=42).fit_predict(cluster_X)
    unique_codes_df['cluster_tmp'] = labels_val
    for cluster_id in sorted(unique_codes_df['cluster_tmp'].unique()):
        members_df = unique_codes_df[unique_codes_df['cluster_tmp'].eq(cluster_id)].sort_values(['responses','frequency','code_clean'], ascending=[False, False, True])
        top_codes = members_df['code_clean'].head(10).tolist()
        cluster_summary_rows.append({'k': k_val, 'cluster': cluster_id, 'size': len(members_df), 'top_codes': '; '.join(top_codes)})
        cluster_members[(k_val, cluster_id)] = members_df['code_clean'].tolist()
cluster_summary_df = pd.DataFrame(cluster_summary_rows)
print(cluster_summary_df[cluster_summary_df['k'].isin([7, 8])])
print(cluster_summary_df[cluster_summary_df['k'].eq(10)])

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from itertools import combinations
import pandas as pd
import numpy as np

agreement_rows = []
for k_val in k_values:
    method_names = ['kmeans','agg_ward','agg_average','spectral','gmm']
    ari_vals = []
    nmi_vals = []
    for method_a, method_b in combinations(method_names, 2):
        labels_a = labels_store[method_a + '_' + str(k_val)]
        labels_b = labels_store[method_b + '_' + str(k_val)]
        ari_vals.append(adjusted_rand_score(labels_a, labels_b))
        nmi_vals.append(normalized_mutual_info_score(labels_a, labels_b))
    agreement_rows.append({
        'k': k_val,
        'mean_ari_between_methods': np.mean(ari_vals),
        'median_ari_between_methods': np.median(ari_vals),
        'mean_nmi_between_methods': np.mean(nmi_vals),
        'median_nmi_between_methods': np.median(nmi_vals)
    })
agreement_df = pd.DataFrame(agreement_rows)
model_selection_df = consensus_by_k.merge(agreement_df, on='k')
for col_val in ['mean_silhouette','mean_calinski','mean_nmi_between_methods','mean_ari_between_methods']:
    model_selection_df[col_val + '_rank'] = model_selection_df[col_val].rank(ascending=False, method='min')
model_selection_df['mean_davies_rank'] = model_selection_df['mean_davies'].rank(ascending=True, method='min')
model_selection_df['consensus_rank_mean'] = model_selection_df[['mean_silhouette_rank','mean_calinski_rank','mean_nmi_between_methods_rank','mean_ari_between_methods_rank','mean_davies_rank']].mean(axis=1)
print(model_selection_df.sort_values('consensus_rank_mean'))

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

for k_val in [7, 8, 10, 12]:
    labels_val = KMeans(n_clusters=k_val, n_init=100, random_state=42).fit_predict(cluster_X)
    temp_df = unique_codes_df.copy()
    temp_df['cluster'] = labels_val
    temp_df['weighted_n'] = temp_df['responses']
    rows_out = []
    for cluster_id in sorted(temp_df['cluster'].unique()):
        cluster_df = temp_df[temp_df['cluster'].eq(cluster_id)].sort_values(['responses','frequency','code_clean'], ascending=[False, False, True])
        rows_out.append({
            'k': k_val,
            'cluster': cluster_id,
            'codes_in_cluster': len(cluster_df),
            'response_mentions_sum': int(cluster_df['responses'].sum()),
            'top_codes': '; '.join(cluster_df['code_clean'].head(12).tolist())
        })
    print(pd.DataFrame(rows_out))
    print('---')

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import numpy as np
import pandas as pd
from itertools import combinations

method_names = ['kmeans','agg_ward','agg_average','spectral','gmm']
consensus_rows = []
consensus_labels_store = {}

for k_val in range(3, 13):
    coassoc = np.zeros((len(code_texts), len(code_texts)))
    for method_name in method_names:
        lab = labels_store[method_name + '_' + str(k_val)]
        coassoc = coassoc + (lab[:, None] == lab[None, :]).astype(float)
    coassoc = coassoc / len(method_names)
    dist = 1 - coassoc
    np.fill_diagonal(dist, 0)
    link = linkage(squareform(dist, checks=False), method='average')
    con_lab = fcluster(link, t=k_val, criterion='maxclust') - 1
    consensus_labels_store[k_val] = con_lab
    ari_vals = []
    nmi_vals = []
    for method_name in method_names:
        lab = labels_store[method_name + '_' + str(k_val)]
        ari_vals.append(adjusted_rand_score(con_lab, lab))
        nmi_vals.append(normalized_mutual_info_score(con_lab, lab))
    consensus_rows.append({
        'k': k_val,
        'consensus_silhouette': silhouette_score(cluster_X, con_lab, metric='cosine'),
        'mean_ari_to_methods': np.mean(ari_vals),
        'mean_nmi_to_methods': np.mean(nmi_vals),
        'min_cluster_size': pd.Series(con_lab).value_counts().min(),
        'max_cluster_size': pd.Series(con_lab).value_counts().max()
    })

consensus_cluster_eval = pd.DataFrame(consensus_rows)
print(consensus_cluster_eval)

In [ ]:
import pandas as pd
import numpy as np

candidate_labels = {}
for k_val in [7, 8, 10, 12]:
    labels_val = consensus_labels_store[k_val]
    temp_codes = unique_codes_df[['code_clean','frequency','responses','reviewers']].copy()
    temp_codes['cluster'] = labels_val
    rows_out = []
    for cluster_id in sorted(temp_codes['cluster'].unique()):
        cluster_df = temp_codes[temp_codes['cluster'].eq(cluster_id)].sort_values(['responses','frequency','code_clean'], ascending=[False, False, True])
        rows_out.append({
            'k': k_val,
            'cluster': int(cluster_id),
            'codes_in_cluster': int(len(cluster_df)),
            'response_mentions_sum': int(cluster_df['responses'].sum()),
            'top_codes': '; '.join(cluster_df['code_clean'].head(18).tolist())
        })
    candidate_labels[k_val] = pd.DataFrame(rows_out)

pd.set_option('display.max_colwidth', 500)
print(candidate_labels[8])
print(candidate_labels[12])

In [ ]:
pd.set_option('display.max_colwidth', 2000)
k_val = 8
labels_val = consensus_labels_store[k_val]
cluster_codes_df = unique_codes_df[['code_clean','frequency','responses','reviewers']].copy()
cluster_codes_df['cluster'] = labels_val
for cluster_id in sorted(cluster_codes_df['cluster'].unique()):
    cluster_df = cluster_codes_df[cluster_codes_df['cluster'].eq(cluster_id)].sort_values(['responses','frequency','code_clean'], ascending=[False, False, True])
    print('Cluster ' + str(cluster_id))
    print('size ' + str(len(cluster_df)))
    print('; '.join(cluster_df['code_clean'].head(30).tolist()))
    print('---')

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

consensus_k = 8
cluster_to_label = {
    0: 'Team dynamics and rapid solution development',
    1: 'Peer learning through exploratory problem solving',
    2: 'Applied AI/ML skill building and onboarding needs',
    3: 'Interdisciplinary collaboration and practical constraints',
    4: 'Educational growth and deliverable clarity',
    5: 'Program resources, support, and feedback structures',
    6: 'Motivation, meaningful impact, and workload constraints',
    7: 'Overall positive teamwork experience and logistics'
}
cluster_to_definition = {
    0: 'Codes describing cohesive team dynamics, smooth communication, rapid ideation, and movement toward feasible clinical or real-world solutions.',
    1: 'Codes describing learning from peers, hands-on exploration, creative problem solving, and scoping challenges such as data limitations or need for early mentorship.',
    2: 'Codes describing practical AI, machine learning, and prototype skill development, alongside requests for more beginner onboarding, time, mentorship, and dataset explanation.',
    3: 'Codes describing interdisciplinary diversity, multiple perspectives, patient-centered or clinical application, and constraints such as time-zone coordination or computing resources.',
    4: 'Codes describing educational and technical learning, exploration of healthcare topics, and confusion or misalignment around deliverable expectations.',
    5: 'Codes describing helpful resources, data access, examples, support, leadership check-ins, feedback, networking, and coordination support.',
    6: 'Codes describing enjoyment, meaningful patient-oriented work, AI exploration, learning as a team, and workload or timeline constraints.',
    7: 'Codes describing broadly positive teamwork and enjoyment while also capturing logistical or submission-related improvement needs.'
}

code_cluster_df = unique_codes_df[['code_clean','frequency','responses','reviewers']].copy()
code_cluster_df['cluster'] = consensus_labels_store[consensus_k]
code_cluster_df['theme'] = code_cluster_df['cluster'].map(cluster_to_label)
code_cluster_df['definition'] = code_cluster_df['cluster'].map(cluster_to_definition)

code_to_theme_label = dict(zip(code_cluster_df['code_clean'], code_cluster_df['theme']))

def response_consensus_themes(code_list):
    return sorted(set([code_to_theme_label[code] for code in code_list if code in code_to_theme_label]))

usable_review_df['consensus_themes_k8'] = usable_review_df['combined_codes_list'].apply(response_consensus_themes)

consensus_theme_rows = []
for cluster_id, theme_name in cluster_to_label.items():
    count_val = usable_review_df['consensus_themes_k8'].apply(lambda vals: theme_name in vals).sum()
    codes_in_cluster = code_cluster_df[code_cluster_df['cluster'].eq(cluster_id)].sort_values(['responses','frequency','code_clean'], ascending=[False, False, True])
    consensus_theme_rows.append({
        'cluster': cluster_id,
        'theme': theme_name,
        'definition': cluster_to_definition[cluster_id],
        'codes_in_cluster': int(len(codes_in_cluster)),
        'responses': int(count_val),
        'percent': round(100 * count_val / len(usable_review_df), 1),
        'top_codes': '; '.join(codes_in_cluster['code_clean'].head(8).tolist())
    })
consensus_theme_summary = pd.DataFrame(consensus_theme_rows).sort_values(['responses','theme'], ascending=[False, True]).reset_index(drop=True)

print(consensus_theme_summary[['theme','responses','percent','codes_in_cluster','top_codes']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

plot_eval_df = consensus_cluster_eval.copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.lineplot(data=plot_eval_df, x='k', y='consensus_silhouette', marker='o', ax=axes[0], color='#4C78A8')
axes[0].set_title('Consensus silhouette by cluster count')
axes[0].set_xlabel('Candidate number of themes')
axes[0].set_ylabel('Silhouette score')
sns.lineplot(data=plot_eval_df, x='k', y='mean_nmi_to_methods', marker='o', ax=axes[1], color='#2E8B57')
axes[1].set_title('Agreement with individual methods')
axes[1].set_xlabel('Candidate number of themes')
axes[1].set_ylabel('Mean normalized mutual information')
plt.tight_layout()
plt.savefig('cluster_diagnostics_theme_count.png', bbox_inches='tight')
plt.show()

plot_consensus = consensus_theme_summary.copy()
plot_consensus['label_text'] = plot_consensus['responses'].astype(str) + ' of 21 (' + plot_consensus['percent'].astype(str) + '%)'
plt.figure(figsize=(12, 7))
sns.barplot(data=plot_consensus, x='responses', y='theme', color='#4C78A8')
for idx_val, row_val in plot_consensus.reset_index(drop=True).iterrows():
    plt.text(row_val['responses'] + 0.15, idx_val, row_val['label_text'], va='center', fontsize=10)
plt.xlabel('Number of responses')
plt.ylabel('Consensus theme')
plt.title('Consensus theme prevalence from clustered reviewer codes')
plt.xlim(0, max(plot_consensus['responses']) + 4)
plt.tight_layout()
plt.savefig('consensus_clustered_theme_prevalence.png', bbox_inches='tight')
plt.show()

consensus_theme_summary.to_csv('consensus_clustered_theme_summary.csv', index=False)
model_selection_df.to_csv('cluster_model_selection_metrics.csv', index=False)
code_cluster_df.to_csv('code_to_consensus_theme_mapping.csv', index=False)

print('cluster_diagnostics_theme_count.png')
print('consensus_clustered_theme_prevalence.png')
print('consensus_clustered_theme_summary.csv')
print('cluster_model_selection_metrics.csv')
print('code_to_consensus_theme_mapping.csv')

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from tqdm import tqdm

if 'usable_review_df' not in globals() or 'consensus_theme_summary' not in globals():
    raise RuntimeError('Expected reviewer coding objects are not in memory. Please rerun the clustering setup cell first.')

sentiment_order_k8 = ['strongly positive', 'moderately positive', 'neutral/mixed']
theme_order_k8 = consensus_theme_summary['theme'].tolist()

sentiment_by_theme_k8 = usable_review_df.explode('consensus_themes_k8').groupby(['consensus_themes_k8', 'combined_sentiment']).size().reset_index(name='count')
full_idx_k8 = pd.MultiIndex.from_product([theme_order_k8, sentiment_order_k8], names=['theme', 'sentiment'])
sentiment_by_theme_k8 = sentiment_by_theme_k8.set_index(['consensus_themes_k8', 'combined_sentiment']).reindex(full_idx_k8, fill_value=0).reset_index()
sentiment_by_theme_k8.columns = ['theme', 'sentiment', 'count']

sentiment_by_theme_pct_k8 = sentiment_by_theme_k8.copy()
theme_totals_k8 = sentiment_by_theme_pct_k8.groupby('theme')['count'].transform('sum')
sentiment_by_theme_pct_k8['percent_within_theme'] = np.where(theme_totals_k8 > 0, sentiment_by_theme_pct_k8['count'] / theme_totals_k8 * 100, 0).round(1)

cooccurrence_k8 = pd.DataFrame(0, index=theme_order_k8, columns=theme_order_k8)
for theme_vals in usable_review_df['consensus_themes_k8']:
    unique_vals = sorted(set(theme_vals))
    for theme_a in unique_vals:
        for theme_b in unique_vals:
            if theme_a != theme_b:
                cooccurrence_k8.loc[theme_a, theme_b] = cooccurrence_k8.loc[theme_a, theme_b] + 1

print(sentiment_by_theme_pct_k8)
print(cooccurrence_k8)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import textwrap

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

short_theme_labels_k8 = {
    'Overall positive teamwork experience and logistics': 'Teamwork and logistics',
    'Interdisciplinary collaboration and practical constraints': 'Interdisciplinary collaboration',
    'Motivation, meaningful impact, and workload constraints': 'Motivation, impact, and workload',
    'Applied AI/ML skill building and onboarding needs': 'AI/ML skill building and onboarding',
    'Program resources, support, and feedback structures': 'Resources, support, and feedback',
    'Educational growth and deliverable clarity': 'Educational growth and clarity',
    'Peer learning through exploratory problem solving': 'Peer learning and problem solving',
    'Team dynamics and rapid solution development': 'Team dynamics and rapid solutions'
}

sent_plot = sentiment_by_theme_k8.copy()
sent_plot['theme_label'] = sent_plot['theme'].map(short_theme_labels_k8)
sent_pivot = sent_plot.pivot(index='theme_label', columns='sentiment', values='count')
sent_pivot = sent_pivot.loc[[short_theme_labels_k8[val] for val in theme_order_k8]]
sent_pivot = sent_pivot[['strongly positive', 'moderately positive', 'neutral/mixed']]

ax_obj = sent_pivot.plot(kind='barh', stacked=True, figsize=(12, 7.5), color=['#2E8B57', '#E3A008', '#C44E52'], edgecolor='none')
ax_obj.set_xlabel('Number of theme-coded responses')
ax_obj.set_ylabel('Consensus theme')
ax_obj.set_title('Sentiment distribution within clustered consensus themes')
ax_obj.legend(title='Sentiment', loc='lower right', frameon=True)
for container_val in ax_obj.containers:
    labels_val = [str(int(val)) if val > 0 else '' for val in container_val.datavalues]
    ax_obj.bar_label(container_val, labels=labels_val, label_type='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('figure4_k8_sentiment_by_theme.png', bbox_inches='tight')
plt.show()

cooc_plot = cooccurrence_k8.loc[theme_order_k8, theme_order_k8].copy()
cooc_plot.index = [short_theme_labels_k8[val] for val in cooc_plot.index]
cooc_plot.columns = [short_theme_labels_k8[val] for val in cooc_plot.columns]
plt.figure(figsize=(12, 9.5))
sns.heatmap(cooc_plot, annot=True, fmt='d', cmap='Blues', linewidths=0.5, linecolor='white', cbar_kws={'label':'Co-occurring responses'})
plt.title('Co-occurrence of clustered consensus themes')
plt.xlabel('Consensus theme')
plt.ylabel('Consensus theme')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('figure5_k8_theme_cooccurrence_heatmap.png', bbox_inches='tight')
plt.show()

sentiment_by_theme_pct_k8.to_csv('k8_sentiment_by_theme.csv', index=False)
cooccurrence_k8.to_csv('k8_theme_cooccurrence_matrix.csv')
print('figure4_k8_sentiment_by_theme.png')
print('figure5_k8_theme_cooccurrence_heatmap.png')
print('k8_sentiment_by_theme.csv')
print('k8_theme_cooccurrence_matrix.csv')

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

cooc_pair_rows = []
response_theme_sets = usable_review_df['consensus_themes_k8'].apply(set).tolist()
theme_counts_k8 = {theme_val: sum(theme_val in theme_set for theme_set in response_theme_sets) for theme_val in theme_order_k8}

for theme_a, theme_b in combinations(theme_order_k8, 2):
    both_count = sum((theme_a in theme_set) and (theme_b in theme_set) for theme_set in response_theme_sets)
    either_count = sum((theme_a in theme_set) or (theme_b in theme_set) for theme_set in response_theme_sets)
    jaccard_val = both_count / either_count if either_count else 0
    cooc_pair_rows.append({
        'theme_a': theme_a,
        'theme_b': theme_b,
        'cooccurring_responses': both_count,
        'jaccard_overlap': round(jaccard_val, 3),
        'theme_a_responses': theme_counts_k8[theme_a],
        'theme_b_responses': theme_counts_k8[theme_b]
    })

k8_cooccurrence_pairs = pd.DataFrame(cooc_pair_rows).sort_values(['cooccurring_responses','jaccard_overlap'], ascending=[False, False]).reset_index(drop=True)
k8_theme_summary_with_sentiment = consensus_theme_summary[['theme','responses','percent','codes_in_cluster','top_codes']].merge(
    sentiment_by_theme_pct_k8.pivot(index='theme', columns='sentiment', values='count').reset_index(),
    on='theme', how='left'
)

k8_cooccurrence_pairs.to_csv('k8_theme_cooccurrence_pairs.csv', index=False)
k8_theme_summary_with_sentiment.to_csv('k8_theme_summary_with_sentiment.csv', index=False)
print(k8_theme_summary_with_sentiment)
print(k8_cooccurrence_pairs.head(12))
print('k8_theme_cooccurrence_pairs.csv')
print('k8_theme_summary_with_sentiment.csv')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

short_theme_labels_k8 = {
    'Overall positive teamwork experience and logistics': 'Teamwork and logistics',
    'Interdisciplinary collaboration and practical constraints': 'Interdisciplinary collaboration',
    'Motivation, meaningful impact, and workload constraints': 'Motivation, impact, and workload',
    'Applied AI/ML skill building and onboarding needs': 'AI/ML skill building and onboarding',
    'Program resources, support, and feedback structures': 'Resources, support, and feedback',
    'Educational growth and deliverable clarity': 'Educational growth and clarity',
    'Peer learning through exploratory problem solving': 'Peer learning and problem solving',
    'Team dynamics and rapid solution development': 'Team dynamics and rapid solutions'
}

theme_order_by_frequency_k8 = consensus_theme_summary.sort_values(['responses', 'theme'], ascending=[False, True])['theme'].tolist()
print(consensus_theme_summary.sort_values(['responses', 'theme'], ascending=[False, True])[['theme', 'responses', 'percent']])

sent_plot = sentiment_by_theme_k8.copy()
sent_plot['theme_label'] = sent_plot['theme'].map(short_theme_labels_k8)
ordered_theme_labels_k8 = [short_theme_labels_k8[val] for val in theme_order_by_frequency_k8]

sent_pivot = sent_plot.pivot(index='theme_label', columns='sentiment', values='count')
sent_pivot = sent_pivot.reindex(ordered_theme_labels_k8)
sent_pivot = sent_pivot[['strongly positive', 'moderately positive', 'neutral/mixed']]
sent_pivot_plot = sent_pivot.iloc[::-1]

ax_obj = sent_pivot_plot.plot(kind='barh', stacked=True, figsize=(12, 7.5), color=['#2E8B57', '#E3A008', '#C44E52'], edgecolor='none')
ax_obj.set_xlabel('Number of theme-coded responses')
ax_obj.set_ylabel('Consensus theme')
ax_obj.set_title('Sentiment distribution within clustered consensus themes')
ax_obj.legend(title='Sentiment', loc='lower right', frameon=True)
for container_val in ax_obj.containers:
    labels_val = [str(int(val)) if val > 0 else '' for val in container_val.datavalues]
    ax_obj.bar_label(container_val, labels=labels_val, label_type='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('figure4_k8_sentiment_by_theme_restyled.png', bbox_inches='tight')
plt.show()

cooc_plot = cooccurrence_k8.loc[theme_order_by_frequency_k8, theme_order_by_frequency_k8].copy()
cooc_plot.index = [short_theme_labels_k8[val] for val in cooc_plot.index]
cooc_plot.columns = [short_theme_labels_k8[val] for val in cooc_plot.columns]
mask_upper = np.triu(np.ones_like(cooc_plot, dtype=bool), k=0)
plt.figure(figsize=(12, 9.5))
sns.heatmap(cooc_plot, mask=mask_upper, annot=True, fmt='d', cmap='Blues', linewidths=0.5, linecolor='white', cbar_kws={'label':'Co-occurring responses'}, square=True)
plt.title('Co-occurrence of clustered consensus themes')
plt.xlabel('Consensus theme')
plt.ylabel('Consensus theme')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('figure5_k8_theme_cooccurrence_heatmap_restyled.png', bbox_inches='tight')
plt.show()

sentiment_by_theme_pct_k8.to_csv('k8_sentiment_by_theme_restyled.csv', index=False)
cooccurrence_k8.loc[theme_order_by_frequency_k8, theme_order_by_frequency_k8].to_csv('k8_theme_cooccurrence_matrix_restyled_order.csv')
print('figure4_k8_sentiment_by_theme_restyled.png')
print('figure5_k8_theme_cooccurrence_heatmap_restyled.png')
print('k8_sentiment_by_theme_restyled.csv')
print('k8_theme_cooccurrence_matrix_restyled_order.csv')

##7 themes

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

seven_theme_map = {
    'Overall positive teamwork experience and logistics': 'Overall positive teamwork experience and logistics',
    'Team dynamics and rapid solution development': 'Overall positive teamwork experience and logistics',
    'Interdisciplinary collaboration and practical constraints': 'Interdisciplinary collaboration and practical constraints',
    'Motivation, meaningful impact, and workload constraints': 'Motivation, meaningful impact, and workload constraints',
    'Applied AI/ML skill building and onboarding needs': 'Applied AI/ML skill building and onboarding needs',
    'Program resources, support, and feedback structures': 'Program resources, support, and feedback structures',
    'Educational growth and deliverable clarity': 'Educational growth and deliverable clarity',
    'Peer learning through exploratory problem solving': 'Peer learning through exploratory problem solving'
}

seven_theme_order = [
    'Overall positive teamwork experience and logistics',
    'Interdisciplinary collaboration and practical constraints',
    'Motivation, meaningful impact, and workload constraints',
    'Applied AI/ML skill building and onboarding needs',
    'Program resources, support, and feedback structures',
    'Educational growth and deliverable clarity',
    'Peer learning through exploratory problem solving'
]

def collapse_to_seven(theme_vals):
    return sorted(set([seven_theme_map[theme_val] for theme_val in theme_vals if theme_val in seven_theme_map]))

usable_review_df['consensus_themes_k7_major'] = usable_review_df['consensus_themes_k8'].apply(collapse_to_seven)

seven_summary_rows = []
for theme_val in seven_theme_order:
    count_val = usable_review_df['consensus_themes_k7_major'].apply(lambda vals: theme_val in vals).sum()
    seven_summary_rows.append({'theme': theme_val, 'responses': int(count_val), 'percent': round(100 * count_val / len(usable_review_df), 1)})
seven_theme_summary = pd.DataFrame(seven_summary_rows).sort_values(['responses', 'theme'], ascending=[False, True]).reset_index(drop=True)

print(seven_theme_summary)

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

sentiment_order_k7 = ['strongly positive', 'moderately positive', 'neutral/mixed']
theme_order_k7 = seven_theme_summary['theme'].tolist()

sentiment_by_theme_k7 = usable_review_df.explode('consensus_themes_k7_major').groupby(['consensus_themes_k7_major', 'combined_sentiment']).size().reset_index(name='count')
full_idx_k7 = pd.MultiIndex.from_product([theme_order_k7, sentiment_order_k7], names=['theme', 'sentiment'])
sentiment_by_theme_k7 = sentiment_by_theme_k7.set_index(['consensus_themes_k7_major', 'combined_sentiment']).reindex(full_idx_k7, fill_value=0).reset_index()
sentiment_by_theme_k7.columns = ['theme', 'sentiment', 'count']
sentiment_by_theme_pct_k7 = sentiment_by_theme_k7.copy()
theme_totals_k7 = sentiment_by_theme_pct_k7.groupby('theme')['count'].transform('sum')
sentiment_by_theme_pct_k7['percent_within_theme'] = np.where(theme_totals_k7 > 0, sentiment_by_theme_pct_k7['count'] / theme_totals_k7 * 100, 0).round(1)

cooccurrence_k7 = pd.DataFrame(0, index=theme_order_k7, columns=theme_order_k7)
for theme_vals in usable_review_df['consensus_themes_k7_major']:
    unique_vals = sorted(set(theme_vals))
    for theme_a in unique_vals:
        for theme_b in unique_vals:
            if theme_a != theme_b:
                cooccurrence_k7.loc[theme_a, theme_b] = cooccurrence_k7.loc[theme_a, theme_b] + 1

print(sentiment_by_theme_pct_k7)
print(cooccurrence_k7)

In [ ]:
import pandas as pd
import numpy as np

team_dynamics_theme = 'Team dynamics and rapid solution development'
team_dynamics_rows = usable_review_df[usable_review_df['consensus_themes_k8'].apply(lambda vals: team_dynamics_theme in vals)].copy()
team_dynamics_sample = team_dynamics_rows[['response_id', 'free_response_clean', 'combined_sentiment', 'consensus_themes_k8', 'consensus_themes_k7_major']].head(3)
print(team_dynamics_sample)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

short_theme_labels_k7 = {
    'Overall positive teamwork experience and logistics': 'Teamwork and logistics',
    'Interdisciplinary collaboration and practical constraints': 'Interdisciplinary collaboration',
    'Motivation, meaningful impact, and workload constraints': 'Motivation, impact, and workload',
    'Applied AI/ML skill building and onboarding needs': 'AI/ML skill building and onboarding',
    'Program resources, support, and feedback structures': 'Resources, support, and feedback',
    'Educational growth and deliverable clarity': 'Educational growth and clarity',
    'Peer learning through exploratory problem solving': 'Peer learning and problem solving'
}

theme_order_by_frequency_k7 = seven_theme_summary.sort_values(['responses', 'theme'], ascending=[False, True])['theme'].tolist()
ordered_theme_labels_k7 = [short_theme_labels_k7[val] for val in theme_order_by_frequency_k7]

print(seven_theme_summary[['theme', 'responses', 'percent']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

sent_plot_k7 = sentiment_by_theme_k7.copy()
sent_plot_k7['theme_label'] = sent_plot_k7['theme'].map(short_theme_labels_k7)
sent_pivot_k7 = sent_plot_k7.pivot(index='theme_label', columns='sentiment', values='count')
sent_pivot_k7 = sent_pivot_k7.reindex(ordered_theme_labels_k7)
sent_pivot_k7 = sent_pivot_k7[['strongly positive', 'moderately positive', 'neutral/mixed']]
sent_pivot_plot_k7 = sent_pivot_k7.iloc[::-1]

ax_obj = sent_pivot_plot_k7.plot(kind='barh', stacked=True, figsize=(12, 7), color=['#2E8B57', '#E3A008', '#C44E52'], edgecolor='none')
ax_obj.set_xlabel('Number of theme-coded responses')
ax_obj.set_ylabel('Major theme')
ax_obj.set_title('Sentiment distribution within 7 major themes')
ax_obj.legend(title='Sentiment', loc='lower right', frameon=True)
for container_val in ax_obj.containers:
    labels_val = [str(int(val)) if val > 0 else '' for val in container_val.datavalues]
    ax_obj.bar_label(container_val, labels=labels_val, label_type='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('figure4_k7_sentiment_by_theme.png', bbox_inches='tight')
plt.show()

cooc_plot_k7 = cooccurrence_k7.loc[theme_order_by_frequency_k7, theme_order_by_frequency_k7].copy()
cooc_plot_k7.index = [short_theme_labels_k7[val] for val in cooc_plot_k7.index]
cooc_plot_k7.columns = [short_theme_labels_k7[val] for val in cooc_plot_k7.columns]
mask_upper_k7 = np.triu(np.ones_like(cooc_plot_k7, dtype=bool), k=0)
plt.figure(figsize=(11, 8.75))
sns.heatmap(cooc_plot_k7, mask=mask_upper_k7, annot=True, fmt='d', cmap='Blues', linewidths=0.5, linecolor='white', cbar_kws={'label':'Co-occurring responses'}, square=True)
plt.title('Co-occurrence of 7 major themes')
plt.xlabel('Major theme')
plt.ylabel('Major theme')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('figure5_k7_theme_cooccurrence_heatmap.png', bbox_inches='tight')
plt.show()

sil_df = consensus_cluster_eval[['k', 'consensus_silhouette']].copy()
sil_df['silhouette_gain_from_prior_k'] = sil_df['consensus_silhouette'].diff()
plt.figure(figsize=(10, 6))
sns.lineplot(data=sil_df, x='k', y='consensus_silhouette', marker='o', color='#4C78A8', linewidth=2.5)
plt.axvline(7, color='#C44E52', linestyle='--', linewidth=2, label='Selected 7-theme solution')
plt.scatter([7], [sil_df.loc[sil_df['k'].eq(7), 'consensus_silhouette'].iloc[0]], color='#C44E52', s=120, zorder=5)
plt.title('Consensus silhouette by candidate number of themes')
plt.xlabel('Candidate number of themes')
plt.ylabel('Consensus silhouette score')
plt.xticks(sil_df['k'].astype(int).tolist())
plt.legend(frameon=True, loc='lower right')
plt.tight_layout()
plt.savefig('figure3_k7_silhouette_elbow.png', bbox_inches='tight')
plt.show()

print(sil_df)